# Model Training on Combined Dataset

This notebook trains models on the combined LIAR + ISOT dataset.

First, make sure you've run `data/preprocess_combined.ipynb` to create the combined datasets.


In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# Load combined datasets
train_df = pd.read_csv('data/processed/train_combined.csv')
valid_df = pd.read_csv('data/processed/valid_combined.csv')
test_df = pd.read_csv('data/processed/test_combined.csv')

print(f"Training set shape: {train_df.shape}")
print(f"Validation set shape: {valid_df.shape}")
print(f"Test set shape: {test_df.shape}")

print(f"\nTraining set label distribution:\n{train_df['label'].value_counts()}")
print(f"\nTraining set dataset distribution:\n{train_df['dataset'].value_counts()}")


Training set shape: (41668, 3)
Validation set shape: (8019, 3)
Test set shape: (8002, 3)

Training set label distribution:
label
0    23038
1    18630
Name: count, dtype: int64

Training set dataset distribution:
dataset
ISOT    31428
LIAR    10240
Name: count, dtype: int64


In [ ]:
# Prepare features and targets
x_train = train_df['text'].fillna('')
y_train = train_df['label']

x_valid = valid_df['text'].fillna('')
y_valid = valid_df['label']

x_test = test_df['text'].fillna('')
y_test = test_df['label']

# LEAKAGE CHECK: Verify we're only using text, not metadata
print("Features being used:")
print(f"  - Training: {x_train.shape[0]} samples")
print(f"  - Only using 'text' column (no subject, no metadata)")
print(f"\nDataset source distribution in training:")
print(train_df['dataset'].value_counts())
print(f"\nLabel distribution by dataset:")
print(train_df.groupby('dataset')['label'].value_counts())


## Logistic Regression Model


In [3]:
# Logistic Regression
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1,2))),
    ('clf', LogisticRegression(solver='liblinear', random_state=42, max_iter=1000))
])

print("Training Logistic Regression...")
lr_pipeline.fit(x_train, y_train)

y_pred_valid = lr_pipeline.predict(x_valid)
print("\n-- Logistic Regression - Validation Set --")
print(classification_report(y_valid, y_pred_valid, target_names=['False', 'True']))

y_pred_test = lr_pipeline.predict(x_test)
print("\n-- Logistic Regression - Test Set --")
print(classification_report(y_test, y_pred_test, target_names=['False', 'True']))


Training Logistic Regression...

-- Logistic Regression - Validation Set --
              precision    recall  f1-score   support

       False       0.92      0.96      0.94      4386
        True       0.95      0.90      0.93      3633

    accuracy                           0.93      8019
   macro avg       0.94      0.93      0.93      8019
weighted avg       0.94      0.93      0.93      8019


-- Logistic Regression - Test Set --
              precision    recall  f1-score   support

       False       0.91      0.97      0.94      4341
        True       0.96      0.89      0.92      3661

    accuracy                           0.93      8002
   macro avg       0.93      0.93      0.93      8002
weighted avg       0.93      0.93      0.93      8002



## SVM Model


In [ ]:
# SVM
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1,2))),
    ('clf', SVC(kernel='linear', random_state=42, class_weight='balanced'))
])

print("Training SVM...")
svm_pipeline.fit(x_train, y_train)

y_pred_valid = svm_pipeline.predict(x_valid)
print("\n-- SVM - Validation Set --")
print(classification_report(y_valid, y_pred_valid, target_names=['False', 'True']))

y_pred_test = svm_pipeline.predict(x_test)
print("\n-- SVM - Test Set --")
print(classification_report(y_test, y_pred_test, target_names=['False', 'True']))


Training SVM...


## Random Forest Model


In [3]:
# Random Forest (optimized for speed)
# Optimizations for large combined dataset:
# - max_features=10000: Limits TF-IDF features to top 10k (reduces dimensionality significantly)
# - n_estimators=50: Fewer trees for faster training (still good performance)
# - max_depth=20: Limits tree depth to prevent overfitting and speed up training
# - max_samples=0.7: Uses 70% of data per tree (faster on large dataset, still robust)
rf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=10000)),
    ('clf', RandomForestClassifier(
        n_estimators=50, 
        max_depth=20,
        max_samples=0.7,
        random_state=42, 
        class_weight='balanced', 
        n_jobs=-1
    ))
])

print("Training Random Forest (optimized for speed)...")
print("Note: This may still take 5-15 minutes due to the large dataset size (~41k samples)")
rf_pipeline.fit(x_train, y_train)

y_pred_valid = rf_pipeline.predict(x_valid)
print("\n-- Random Forest - Validation Set --")
print(classification_report(y_valid, y_pred_valid, target_names=['False', 'True']))

y_pred_test = rf_pipeline.predict(x_test)
print("\n-- Random Forest - Test Set --")
print(classification_report(y_test, y_pred_test, target_names=['False', 'True']))


Training Random Forest (optimized for speed)...
Note: This may still take 5-15 minutes due to the large dataset size (~41k samples)

-- Random Forest - Validation Set --
              precision    recall  f1-score   support

       False       0.91      0.99      0.95      4386
        True       0.99      0.88      0.93      3633

    accuracy                           0.94      8019
   macro avg       0.95      0.94      0.94      8019
weighted avg       0.95      0.94      0.94      8019


-- Random Forest - Test Set --
              precision    recall  f1-score   support

       False       0.90      0.99      0.95      4341
        True       0.99      0.88      0.93      3661

    accuracy                           0.94      8002
   macro avg       0.95      0.93      0.94      8002
weighted avg       0.94      0.94      0.94      8002



## Leakage Analysis

Let's check if the high scores are due to data leakage or legitimate performance.


In [ ]:
# Check for potential leakage sources
# 1. Check if dataset source (LIAR vs ISOT) is predictive
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Check if we can predict dataset source from text (this shouldn't be too easy)
print("=== Leakage Check 1: Can we predict dataset source from text? ===")
print("(If this is too easy, the model might be learning dataset-specific patterns)\n")

# Create dataset source labels
dataset_train = train_df['dataset'].map({'LIAR': 0, 'ISOT': 1})
dataset_valid = valid_df['dataset'].map({'LIAR': 0, 'ISOT': 1})

leakage_check = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=5000)),
    ('clf', LogisticRegression(solver='liblinear', random_state=42, max_iter=500))
])

leakage_check.fit(x_train, dataset_train)
pred_dataset = leakage_check.predict(x_valid)
print(classification_report(dataset_valid, pred_dataset, target_names=['LIAR', 'ISOT']))

print("\n=== Interpretation ===")
print("If accuracy > 0.8, the datasets have distinct text patterns.")
print("This is expected but means the model might learn dataset-specific features.")
print("However, this doesn't necessarily mean leakage - it's just that datasets differ.\n")


In [ ]:
# Check 2: Performance on each dataset separately
print("=== Leakage Check 2: Performance on LIAR vs ISOT separately ===\n")

# Test on LIAR-only
liar_valid = valid_df[valid_df['dataset'] == 'LIAR']
liar_test = test_df[test_df['dataset'] == 'LIAR']

if len(liar_valid) > 0:
    x_liar_valid = liar_valid['text'].fillna('')
    y_liar_valid = liar_valid['label']
    
    y_pred_liar = lr_pipeline.predict(x_liar_valid)
    print("Logistic Regression on LIAR validation set:")
    print(classification_report(y_liar_valid, y_pred_liar, target_names=['False', 'True']))

# Test on ISOT-only  
isot_valid = valid_df[valid_df['dataset'] == 'ISOT']
isot_test = test_df[test_df['dataset'] == 'ISOT']

if len(isot_valid) > 0:
    x_isot_valid = isot_valid['text'].fillna('')
    y_isot_valid = isot_valid['label']
    
    y_pred_isot = lr_pipeline.predict(x_isot_valid)
    print("\nLogistic Regression on ISOT validation set:")
    print(classification_report(y_isot_valid, y_pred_isot, target_names=['False', 'True']))

print("\n=== Interpretation ===")
print("If performance is much better on one dataset, there might be dataset-specific patterns.")
print("This is normal when combining datasets, but worth noting.")


In [ ]:
# Check 3: Cross-dataset performance (train on one, test on other)
print("=== Leakage Check 3: Cross-dataset generalization ===\n")
print("This tests if the model generalizes across datasets.\n")

# Train on LIAR only, test on ISOT
liar_train_only = train_df[train_df['dataset'] == 'LIAR']
x_liar_train = liar_train_only['text'].fillna('')
y_liar_train = liar_train_only['label']

cross_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=10000)),
    ('clf', LogisticRegression(solver='liblinear', random_state=42, max_iter=1000))
])

print("Training on LIAR only, testing on ISOT validation:")
cross_pipeline.fit(x_liar_train, y_liar_train)
if len(isot_valid) > 0:
    y_pred_cross = cross_pipeline.predict(x_isot_valid)
    print(classification_report(y_isot_valid, y_pred_cross, target_names=['False', 'True']))

print("\n=== Interpretation ===")
print("If cross-dataset performance drops significantly (< 0.7), the model is learning")
print("dataset-specific patterns rather than general fake news patterns.")
print("This suggests the high scores might be partially due to dataset memorization.")
